# 1. Load Superstore Dataset

In [1]:
import sys

print("RUNNING WITH:", sys.executable)
import pandas as pd

file_path = "./data/Sample - Superstore.csv"

df = pd.read_csv(file_path, encoding="latin-1")

RUNNING WITH: c:\python314\python.exe


# 2. Basic inspection

In [2]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst few rows:")
print(df.head())

Dataset shape: (9994, 21)

Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

First few rows:
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderso

# 3. Clean product names

In [3]:
df["Product Name"] = df["Product Name"].astype(str).str.strip()

# 4. Aggregate total sales by product

In [5]:
product_sales = df.groupby("Product Name", as_index=False)["Sales"].sum()


product_sales = product_sales.rename(columns={"Sales": "Total Sales"})

# 5. Sort descending by total sales

In [6]:
product_sales = product_sales.sort_values(
    by="Total Sales", ascending=False
).reset_index(drop=True)

# 6. Add rank

In [7]:
product_sales["Rank"] = (
    product_sales["Total Sales"].rank(method="min", ascending=False).astype(int)
)

# 7. Add percentage of total sales

In [8]:
grand_total_sales = product_sales["Total Sales"].sum()

product_sales["% of Total Sales"] = (
    product_sales["Total Sales"] / grand_total_sales * 100
).round(2)

# 8. Strict Top 10

In [9]:
top10_strict = product_sales.head(10).copy()

print("\nStrict Top 10 Products by Sales:")
print(top10_strict[["Rank", "Product Name", "Total Sales", "% of Total Sales"]])


Strict Top 10 Products by Sales:
   Rank                                       Product Name  Total Sales  \
0     1              Canon imageCLASS 2200 Advanced Copier    61599.824   
1     2  Fellowes PB500 Electric Punch Plastic Comb Bin...    27453.384   
2     3  Cisco TelePresence System EX90 Videoconferenci...    22638.480   
3     4       HON 5400 Series Task Chairs for Big and Tall    21870.576   
4     5         GBC DocuBind TL300 Electric Binding System    19823.479   
5     6   GBC Ibimaster 500 Manual ProClick Binding System    19024.500   
6     7               Hewlett Packard LaserJet 3310 Copier    18839.686   
7     8  HP Designjet T520 Inkjet Large Format Printer ...    18374.895   
8     9          GBC DocuBind P400 Electric Binding System    17965.068   
9    10        High Speed Automatic Electric Letter Opener    17030.312   

   % of Total Sales  
0              2.68  
1              1.20  
2              0.99  
3              0.95  
4              0.86  
5       

# 9. Tie check around rank 10

In [10]:
cutoff_sales = product_sales.loc[9, "Total Sales"]


tie_count_at_cutoff = (product_sales["Total Sales"] == cutoff_sales).sum()

print("\nCutoff sales value for Top 10:", cutoff_sales)
print("Number of products tied at cutoff:", tie_count_at_cutoff)

tied_products = product_sales[product_sales["Total Sales"] == cutoff_sales].copy()

print("\nProducts tied at cutoff:")
print(tied_products[["Rank", "Product Name", "Total Sales", "% of Total Sales"]])


Cutoff sales value for Top 10: 17030.311999999998
Number of products tied at cutoff: 1

Products tied at cutoff:
   Rank                                 Product Name  Total Sales  \
9    10  High Speed Automatic Electric Letter Opener    17030.312   

   % of Total Sales  
9              0.74  


# 10. Top 10 including ties

In [11]:
top10_including_ties = product_sales[
    product_sales["Total Sales"] >= cutoff_sales
].copy()

print("\nTop 10 including ties:")
print(top10_including_ties[["Rank", "Product Name", "Total Sales", "% of Total Sales"]])

print("\nNumber of products in strict Top 10:", len(top10_strict))
print("Number of products in Top 10 including ties:", len(top10_including_ties))


Top 10 including ties:
   Rank                                       Product Name  Total Sales  \
0     1              Canon imageCLASS 2200 Advanced Copier    61599.824   
1     2  Fellowes PB500 Electric Punch Plastic Comb Bin...    27453.384   
2     3  Cisco TelePresence System EX90 Videoconferenci...    22638.480   
3     4       HON 5400 Series Task Chairs for Big and Tall    21870.576   
4     5         GBC DocuBind TL300 Electric Binding System    19823.479   
5     6   GBC Ibimaster 500 Manual ProClick Binding System    19024.500   
6     7               Hewlett Packard LaserJet 3310 Copier    18839.686   
7     8  HP Designjet T520 Inkjet Large Format Printer ...    18374.895   
8     9          GBC DocuBind P400 Electric Binding System    17965.068   
9    10        High Speed Automatic Electric Letter Opener    17030.312   

   % of Total Sales  
0              2.68  
1              1.20  
2              0.99  
3              0.95  
4              0.86  
5              0.8

# 11. Supporting findings

In [12]:
highest_product = product_sales.iloc[0]
tenth_product = product_sales.iloc[9]

difference_rank1_vs_rank10 = (
    highest_product["Total Sales"] - tenth_product["Total Sales"]
)

combined_top10_sales = top10_strict["Total Sales"].sum()
combined_top10_percentage = combined_top10_sales / grand_total_sales * 100

print("\n--- Supporting Findings ---")
print("Highest-selling product:", highest_product["Product Name"])
print("Highest product total sales:", highest_product["Total Sales"])

print("10th-ranked product:", tenth_product["Product Name"])
print("10th product total sales:", tenth_product["Total Sales"])

print("Difference between rank 1 and rank 10:", difference_rank1_vs_rank10)

print("Combined Top 10 sales:", combined_top10_sales)
print("Combined Top 10 % of total sales:", round(combined_top10_percentage, 2))


--- Supporting Findings ---
Highest-selling product: Canon imageCLASS 2200 Advanced Copier
Highest product total sales: 61599.824
10th-ranked product: High Speed Automatic Electric Letter Opener
10th product total sales: 17030.311999999998
Difference between rank 1 and rank 10: 44569.512
Combined Top 10 sales: 244620.204
Combined Top 10 % of total sales: 10.65


# 12. Export results for Excel comparison

In [15]:
top10_strict.to_csv("./cross-check/python_top10_strict.csv", index=False)
top10_including_ties.to_csv(
    "./cross-check/python_top10_including_ties.csv", index=False
)
product_sales.to_csv("./cross-check/python_all_product_sales.csv", index=False)

print("\nCSV files exported:")
print("1. python_top10_strict.csv")
print("2. python_top10_including_ties.csv")
print("3. python_all_product_sales.csv")


CSV files exported:
1. python_top10_strict.csv
2. python_top10_including_ties.csv
3. python_all_product_sales.csv
